# SWAN Model Runner: Setup, Execution & Output Parsing

---

## Overview

The **SWAN** (Simulating WAves Nearshore) model is a third-generation spectral wave model developed at Delft University of Technology. It solves the spectral action balance equation to compute wave parameters in coastal waters, accounting for:

- Wind generation
- Whitecapping dissipation
- Bottom friction
- Depth-induced breaking
- Refraction and shoaling
- Nonlinear wave-wave interactions

This notebook provides a **Python interface** for:

1. **Modifying SWAN input files** to inject custom output commands
2. **Invoking the SWAN executable** via `subprocess`
3. **Parsing SWAN output files** (TABLE text and BLOCK `.MAT` binary formats)

These utilities are used by the coupled site-selection and optimization notebooks later in the workflow.

### Prerequisites

- SWAN must be compiled and available on your system PATH (see the repository README for installation instructions)
- Dependencies: `numpy`, `scipy` (for `.MAT` files)

### SWAN Input File Structure

A SWAN input file (`INPUT`) is a sequence of keyword commands:

```
PROJECT 'name' ...
CGRID ...          ! Computational grid
INPgrid BOTtom ... ! Bathymetry input specification
READinp BOTtom ... ! Read bathymetry file
BOUnd ...          ! Boundary conditions
GEN3 ...           ! Physics: wave generation
...
COMPUTE ...        ! Run the simulation
STOP
```

Our runner inserts output commands (TABLE, BLOCK, POINTS) **just before the COMPUTE line**.

In [1]:
import os
import re
import shutil
import subprocess
import numpy as np

## 1. Prepare SWAN Input File

The `prepare_input` function takes an existing SWAN input template, copies it, and inserts additional commands (e.g., output requests) immediately before the `COMPUTE` line. This allows us to programmatically request specific outputs without manually editing the input file each time.

**How it works:**
1. Read all lines from the template
2. Find the line starting with `COMP` (the COMPUTE command)
3. Insert the new commands just before it
4. Write the modified file

In [ ]:
def prepare_input(template_path, output_path, extra_commands):
    """
    Copy a SWAN input template and insert extra commands before COMPUTE.

    Raises FileNotFoundError if the template is missing or ValueError if
    the template has no COMPUTE line.
    """
    if not os.path.exists(template_path):
        raise FileNotFoundError(f"SWAN template not found: {template_path}")
    with open(template_path) as f:
        lines = f.readlines()

    idx = next(
        (i for i, l in enumerate(lines) if l.strip().upper().startswith('COMP')),
        None,
    )
    if idx is None:
        raise ValueError("COMPUTE not found in SWAN input file")

    insert = ['!\n', '!--- output added by swan_runner ---\n']
    for cmd in extra_commands:
        insert.append(cmd.rstrip('\n') + '\n')
    insert.append('!\n')

    lines = lines[:idx] + insert + lines[idx:]
    with open(output_path, 'w') as f:
        f.writelines(lines)
    print(f"Prepared: {output_path}")
    return output_path


## 2. Write Points File

SWAN can output results at specific geographic locations defined in a **points file**. Each line contains a longitude-latitude pair. This is used by the AHP-TOPSIS site-selection workflow to extract wave parameters at candidate deployment sites.

In [ ]:
def write_points_file(filepath, locations):
    """Write a SWAN-format points file (one `lon lat` per line)."""
    locations = list(locations)
    if not locations:
        raise ValueError("At least one (lon, lat) pair is required.")
    with open(filepath, 'w') as f:
        for lon, lat in locations:
            f.write(f'{float(lon):.6f} {float(lat):.6f}\n')
    print(f"Wrote {len(locations)} points to {filepath}")
    return filepath


## 3. Execute SWAN

The `run_swan` function invokes the SWAN executable as a subprocess. Key parameters:

| Parameter | Description |
|-----------|-------------|
| `model_dir` | Working directory containing all SWAN files (INPUT, bathymetry, boundary files, etc.) |
| `input_name` | Name of the input file (without extension) |
| `swan_cmd` | The SWAN executable command (`swanrun`, `swan.exe`, etc.) |
| `nproc` | Number of OpenMP threads for parallel execution |
| `timeout` | Maximum runtime in seconds before the process is killed |

After execution, SWAN writes a `norm_end` file indicating normal completion. If this file is missing and the return code is non-zero, the function raises an error.

In [ ]:
import shlex

def run_swan(model_dir, input_name='INPUT', swan_cmd='swanrun', nproc=4,
             timeout=7200):
    """
    Execute SWAN in ``model_dir`` with ``shell=False`` (safe) and a validated
    thread count. Raises a RuntimeError if SWAN did not complete normally.
    """
    if not os.path.isdir(model_dir):
        raise ValueError(f"model_dir does not exist: {model_dir}")
    if not isinstance(nproc, int) or nproc < 1 or nproc > 256:
        raise ValueError(f"nproc must be an int in [1, 256]; got {nproc!r}")
    if shutil.which(swan_cmd) is None:
        raise FileNotFoundError(f"SWAN command {swan_cmd!r} not found on PATH.")

    base = input_name[:-4] if input_name.lower().endswith('.swn') else input_name
    swn_file = os.path.join(model_dir, f'{base}.swn')
    bare_file = os.path.join(model_dir, base)
    if not os.path.exists(swn_file) and os.path.exists(bare_file):
        shutil.copy2(bare_file, swn_file)

    argv = [swan_cmd, '-input', base]
    if nproc > 1:
        argv += ['-omp', str(nproc)]

    # Delete markers from a previous run so they cannot mask a failure
    norm_end = os.path.join(model_dir, 'norm_end')
    erf_file = os.path.join(model_dir, f'{base}.erf')
    for stale in (norm_end, erf_file):
        if os.path.exists(stale):
            os.remove(stale)

    print(f"    $ {' '.join(shlex.quote(a) for a in argv)}   (in {model_dir})")
    result = subprocess.run(
        argv, cwd=model_dir,
        capture_output=True, text=True, timeout=timeout, check=False,
    )

    if not os.path.exists(norm_end):
        tail = ((result.stderr or '') + '\n' + (result.stdout or '')
                ).strip().splitlines()[-30:]
        raise RuntimeError(
            f"SWAN did not finish normally (rc={result.returncode}, no norm_end) "
            f"in {model_dir}:\n" + "\n".join(tail)
            + f"\nInspect {base}.prt / {base}.erf for details."
        )
    with open(norm_end) as f:
        print(f"    SWAN: {f.read().strip()}")
    if os.path.exists(erf_file) and os.path.getsize(erf_file) > 0:
        print(f"    WARNING: SWAN wrote {os.path.basename(erf_file)} -- check it.")
    return result


## 4. Output Readers

SWAN produces two main output formats:

### 4.1 TABLE Output

A text file with `%`-prefixed header lines followed by space-delimited numerical data. Each row corresponds to a point at a timestep. The variable names (e.g., `Hsig`, `RTpeak`, `Dir`, `Depth`) are extracted from the header.

In [ ]:
_SKIP = ('Run:', 'Table:', 'SWAN', 'Project', 'Variables')

def read_table(filepath):
    """
    Parse a SWAN TABLE output file. Returns {variable: np.ndarray}.

    Robust across SWAN versions: prefers an explicit "% Variables: ..." line,
    falls back to the last non-units header line.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"SWAN TABLE not found: {filepath}")
    headers, data_lines = [], []
    with open(filepath) as f:
        for line in f:
            s = line.strip()
            if s.startswith('%'):
                headers.append(s)
            elif s:
                data_lines.append(s)
    if not data_lines:
        raise ValueError(f"SWAN TABLE has no data rows: {filepath}")

    var_names = None
    for h in headers:
        tail = h.lstrip('%').strip()
        if tail.lower().startswith('variables:'):
            var_names = tail.split(':', 1)[1].split()
            break
    if var_names is None:
        for h in reversed(headers):
            tokens = h.lstrip('%').split()
            if not tokens or '[' in h:
                continue
            if any(t.startswith(_SKIP) for t in tokens):
                continue
            var_names = tokens
            break
    if var_names is None:
        raise ValueError(f"Could not identify variable names in {filepath}")

    data = np.loadtxt(data_lines)
    if data.ndim == 1:
        data = data.reshape(1, -1)
    if data.shape[1] < len(var_names):
        raise ValueError(
            f"Column count {data.shape[1]} < variable count {len(var_names)}"
        )
    var_names = var_names[: data.shape[1]]
    return {v: data[:, i] for i, v in enumerate(var_names)}


### 4.2 BLOCK Output (`.MAT` format)

SWAN can write 2-D gridded output in MATLAB `.MAT` format (via `LAY-OUT 3`). Each variable at each timestep is stored as a separate matrix with a name like `Hsig_20240101_030000`. We use `scipy.io.loadmat` to read these files and a helper function to stack all timesteps into a 3-D array `(time, ny, nx)`.

In [ ]:
def read_mat_block(filepath):
    """Read a SWAN .MAT BLOCK output file via scipy.io.loadmat."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"SWAN BLOCK file not found: {filepath}")
    import scipy.io
    return scipy.io.loadmat(filepath)


def extract_block_variable(mat_data, prefix):
    """
    Collect all timestep arrays for a variable prefix (e.g. 'Hsig')
    and return a 3-D array of shape (time, ny, nx).

    Raises ValueError if timestep frames disagree in shape — usually a
    sign that SWAN stopped mid-way; inspect the .prt log.
    """
    pattern = re.compile(rf'^{re.escape(prefix)}_\d{{8}}_\d{{6}}$')
    keys = sorted(k for k in mat_data
                  if isinstance(k, str) and pattern.match(k))
    if not keys:
        return None
    shape0 = np.asarray(mat_data[keys[0]]).shape
    frames = []
    for k in keys:
        arr = np.asarray(mat_data[k])
        if arr.shape != shape0:
            raise ValueError(
                f"Frame {k!r} has shape {arr.shape}, expected {shape0}. "
                "The SWAN run likely terminated mid-way."
            )
        frames.append(arr)
    return np.stack(frames, axis=0)


## 5. Utility: Check SWAN Availability

Before attempting to run SWAN, we can check whether the executable is found on the system PATH.

In [7]:
def check_swan_available(swan_cmd='swanrun'):
    """Return True if SWAN executable is found on PATH."""
    return shutil.which(swan_cmd) is not None

# Quick check
if check_swan_available():
    print("SWAN is available on your system.")
else:
    print("WARNING: 'swanrun' not found on PATH.")
    print("The coupled notebooks (site selection, GA optimization) require SWAN.")
    print("See the repository README for installation instructions.")

SWAN is available on your system.


## 6. Example: Inspect a SWAN Input File

Let's read the California Bay SWAN input file to see its structure. This helps understand what grid, physics, and boundary settings are configured.

In [ ]:
# Demonstrate the utilities by reading the SWAN input for a selected region.
import sys as _sys
_search = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_search, 'code')):
        REPO_ROOT = _search
        break
    _search = os.path.dirname(_search)
else:
    raise FileNotFoundError('Could not locate repo root')
if os.path.join(REPO_ROOT, 'code') not in _sys.path:
    _sys.path.insert(0, os.path.join(REPO_ROOT, 'code'))

from regions import get_region, list_regions, summarize  # noqa: E402

REGION = os.environ.get('SWAN_REGION', 'california_bay')
cfg = get_region(REGION)
print(summarize(REGION))

input_path = cfg.swan_input_path
if os.path.exists(input_path):
    with open(input_path) as f:
        content = f.read()
    print("=" * 65)
    print(f"SWAN input file for {cfg.display_name}: {os.path.basename(input_path)}")
    print("=" * 65)
    print(content)
else:
    print(f"Input file not found at: {input_path}")

# Per-region figures directory for this notebook
FIGS_DIR = cfg.figs_dir('02_model_setup')
print(f'Figures will be saved to: {FIGS_DIR}')


## Summary

This notebook defined the core SWAN interface functions:

| Function | Purpose |
|----------|--------|
| `prepare_input()` | Inject output commands into a SWAN input template |
| `write_points_file()` | Create a SWAN-format points file for output locations |
| `run_swan()` | Execute the SWAN model as a subprocess |
| `read_table()` | Parse SWAN TABLE text output |
| `read_mat_block()` | Read SWAN BLOCK `.MAT` binary output |
| `extract_block_variable()` | Stack timestep matrices into a 3-D array |
| `check_swan_available()` | Verify SWAN is installed |

These functions are imported by the downstream notebooks:
- `03_site_selection/swan_coupled_ahp_topsis.ipynb`
- `04_optimization/swan_coupled_ga_array.ipynb`